# Week 23, Day 1: Bifurcations in one dimension: saddle-nodes and hysteresis

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w023/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Sweep up, sweep down

Solve 1/(r(1 − r)) = w for the two fold rates and compute the fold inputs `I_folds` (sorted). Then sweep I slowly from −3 to 3 over 20 s and back over another 20 s (Euler, dt = 0.5 ms), and record the inputs `I_up` and `I_down` where r first crosses 0.5 on each sweep.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
tau, w = 10.0, 8.0
F = lambda u: 1 / (1 + math.exp(-(u - 4)))

r_f = np.array([(1 - np.sqrt(1 - 4 / w)) / 2, (1 + np.sqrt(1 - 4 / w)) / 2])
I_folds = np.sort(...)                            # I = 4 + ln(r / (1 - r)) - w r at each fold rate

dt, n = 0.5, 40000
I_sweep_up = np.linspace(-3, 3, n); I_sweep_down = I_sweep_up[::-1]
r = 0.0; r_up, r_down = [], []
for I in I_sweep_up:
    r += dt / tau * (-r + F(w * r + I)); r_up.append(r)
for I in I_sweep_down:
    r += dt / tau * (-r + F(w * r + I)); r_down.append(r)
r_up, r_down = np.array(r_up), np.array(r_down)
I_up = ...
I_down = ...
print(f"folds at I = {I_folds.round(4)};  jump up at {I_up:.3f}, jump down at {I_down:.3f}")
plt.plot(I_sweep_up, r_up, label="input rising"); plt.plot(I_sweep_down, r_down, label="input falling")
plt.xlabel("I"); plt.ylabel("r"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_folds():
    "I_folds are the inputs at the two folds, about -1.066 and 1.066"
    assert abs(I_folds[0] + 1.0657) < 1e-3 and abs(I_folds[1] - 1.0657) < 1e-3

def test_jumps():
    "the jumps found by the slow sweeps are within 0.15 of the folds, up at the right fold and down at the left"
    assert abs(I_up - I_folds[1]) < 0.15 and abs(I_down - I_folds[0]) < 0.15

test_folds()
test_jumps()
print("All checks passed")

### Exercise: Recovery slows near the fold

For r = −1, −0.25, −0.0625 and −0.015625, start ẋ = r + x² at the stable point plus 0.001·√(−r), simulate with Euler steps of 0.001 times the predicted recovery time for 3 recovery times, and fit the decay of log|x − x*| to get `t_rec`. Then fit log t_rec against log(−r) to get the exponent `expo`.

In [ ]:
rs = np.array([-1, -0.25, -0.0625, -0.015625])
t_rec = []
for r_ in rs:
    xs = -np.sqrt(-r_); T_pred = 1 / (2 * np.sqrt(-r_)); dt = 0.001 * T_pred
    x = xs + 0.001 * np.sqrt(-r_); d = []
    for _ in range(3000):
        x += dt * (r_ + x ** 2); d.append(abs(x - xs))
    ts = np.arange(1, 3001) * dt
    t_rec.append(...)
t_rec = np.array(t_rec)
expo = ...
print("recovery times", t_rec.round(3), " predicted", (1 / (2 * np.sqrt(-rs))).round(3), f";  exponent {expo:.3f}")

In [ ]:
import numpy as np
def test_times():
    "measured recovery times are within 3% of 1 / (2 sqrt(-r))"
    assert np.all(np.abs(t_rec - 1 / (2 * np.sqrt(-rs))) * 2 * np.sqrt(-rs) < 0.03)

def test_slope():
    "on a log-log plot the recovery time scales with exponent -0.5 (within 0.03)"
    assert abs(expo + 0.5) < 0.03

test_times()
test_slope()
print("All checks passed")

### Exercise: The square-root f-I curve

For I = 0.0025, 0.01, 0.04, 0.16 and 0.64, start θ at −π and integrate with Euler steps of 0.001 until it passes π twice; store the time between the two crossings in `periods`.

In [ ]:
Is = np.array([0.0025, 0.01, 0.04, 0.16, 0.64])
periods = []
for I in Is:
    th, t, dt, crossings = -np.pi, 0.0, 0.001, []
    while len(crossings) < 2:
        th += ...
        t += dt
        if th > np.pi:
            th -= 2 * np.pi; crossings.append(t)
    periods.append(crossings[1] - crossings[0])
periods = np.array(periods)
print("periods", periods.round(3), " predicted", (np.pi / np.sqrt(Is)).round(3))
plt.plot(Is, 1 / periods, "o-"); plt.xlabel("I"); plt.ylabel("rate (1 / period)"); plt.show()

In [ ]:
import numpy as np
def test_period():
    "simulated periods are within 1% of pi / sqrt(I)"
    assert np.all(np.abs(periods - np.pi / np.sqrt(Is)) / (np.pi / np.sqrt(Is)) < 0.01)

def test_sqrt():
    "the rate scales as I to the power 0.5 (within 0.02)"
    assert abs(np.polyfit(np.log(Is), np.log(1 / periods), 1)[0] - 0.5) < 0.02

test_period()
test_sqrt()
print("All checks passed")